In [5]:
import pandas as pd

flights = pd.read_csv('/kaggle/input/datasets/organizations/usdot/flight-delays/flights.csv', low_memory=False)

print(f"Shape: {flights.shape}")
print(f"\nColumns: {list(flights.columns)}")

# Check delay signal by key factors
print(f"\nAverage departure delay by airline:")
print(flights.groupby('AIRLINE')['DEPARTURE_DELAY'].mean().sort_values(ascending=False))

print(f"\nAverage departure delay by month:")
print(flights.groupby('MONTH')['DEPARTURE_DELAY'].mean())

print(f"\nDelay cause columns check:")
delay_cols = ['AIR_SYSTEM_DELAY','SECURITY_DELAY','AIRLINE_DELAY',
              'LATE_AIRCRAFT_DELAY','WEATHER_DELAY']
for col in delay_cols:
    if col in flights.columns:
        print(f"{col}: {flights[col].notna().sum()} non-null values")

# Check tail number for cascade feasibility
print(f"\nTail number check:")
print(f"Unique tail numbers: {flights['TAIL_NUMBER'].nunique()}")
sample_tail = flights['TAIL_NUMBER'].dropna().iloc[0]
print(f"\nSample flights for one aircraft ({sample_tail}):")
print(flights[flights['TAIL_NUMBER']==sample_tail][
    ['TAIL_NUMBER','ORIGIN_AIRPORT','DESTINATION_AIRPORT',
     'SCHEDULED_DEPARTURE','DEPARTURE_DELAY']
].head(10))

ModuleNotFoundError: No module named 'pandas'

In [5]:
import pandas as pd

# Keep only what we need
cols = ['AIRLINE', 'MONTH', 'DAY_OF_WEEK', 'TAIL_NUMBER',
        'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT',
        'SCHEDULED_DEPARTURE', 'DEPARTURE_DELAY',
        'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
        'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'CANCELLED']

df = flights[cols].copy()

# Drop cancelled flights for now (handle separately later if needed)
df = df[df['CANCELLED'] == 0]

# Categorize time of day from scheduled departure (HHMM format)
def get_time_period(t):
    try:
        t = int(t)
        hour = t // 100
        if 5 <= hour < 12: return 'Morning'
        elif 12 <= hour < 17: return 'Afternoon'
        elif 17 <= hour < 21: return 'Evening'
        else: return 'Night'
    except:
        return None

df['TIME_PERIOD'] = df['SCHEDULED_DEPARTURE'].apply(get_time_period)

# Categorize delay into target classes
def categorize_delay(d):
    if pd.isna(d): return None
    if d <= 15: return 'OnTime'      # includes early/slightly late
    elif d <= 60: return 'Short'
    else: return 'Long'

df['DELAY_CATEGORY'] = df['DEPARTURE_DELAY'].apply(categorize_delay)

df.dropna(subset=['DELAY_CATEGORY', 'TIME_PERIOD'], inplace=True)

print(f"Clean shape: {df.shape}")
print(f"\nDelay category distribution:\n{df['DELAY_CATEGORY'].value_counts()}")
print(f"\nTime period distribution:\n{df['TIME_PERIOD'].value_counts()}")

Clean shape: (5729195, 16)

Delay category distribution:
DELAY_CATEGORY
OnTime    4712659
Short      693543
Long       322993
Name: count, dtype: int64

Time period distribution:
TIME_PERIOD
Morning      2347505
Afternoon    1725487
Evening      1290848
Night         365355
Name: count, dtype: int64


In [6]:
import pandas as pd

# Balance via undersampling — keep it real, no synthetic data
min_class_size = df['DELAY_CATEGORY'].value_counts().min()  # Long = 322,993

# Sample equal amounts from each class (cap for speed — 100k per class is plenty)
sample_size = min(min_class_size, 100000)

df_balanced = (
    df.groupby('DELAY_CATEGORY', group_keys=False)
    .apply(lambda x: x.sample(n=sample_size, random_state=42))
)

print(f"Balanced shape: {df_balanced.shape}")
print(f"\nDelay category distribution:\n{df_balanced['DELAY_CATEGORY'].value_counts()}")

# Check signal still holds after balancing
print(f"\nDelay category by airline (top 5 rows):")
print(pd.crosstab(df_balanced['AIRLINE'], df_balanced['DELAY_CATEGORY'], normalize='index').head())

print(f"\nDelay category by time period:")
print(pd.crosstab(df_balanced['TIME_PERIOD'], df_balanced['DELAY_CATEGORY'], normalize='index'))

Balanced shape: (300000, 16)

Delay category distribution:
DELAY_CATEGORY
Long      100000
OnTime    100000
Short     100000
Name: count, dtype: int64

Delay category by airline (top 5 rows):
DELAY_CATEGORY      Long    OnTime     Short
AIRLINE                                     
AA              0.330446  0.347302  0.322252
AS              0.226732  0.495575  0.277693
B6              0.385713  0.280460  0.333826
DL              0.279449  0.414130  0.306421
EV              0.372349  0.337845  0.289806

Delay category by time period:
DELAY_CATEGORY      Long    OnTime     Short
TIME_PERIOD                                 
Afternoon       0.337149  0.305307  0.357544
Evening         0.397798  0.235595  0.366607
Morning         0.264309  0.468085  0.267606
Night           0.358722  0.264551  0.376728


In [7]:
!pip install pgmpy

In [8]:
import pickle
import warnings
warnings.filterwarnings('ignore')

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm

# ── Keep only BN-relevant columns ──────────────────────────────────
bn_df = df_balanced[['AIRLINE', 'MONTH', 'TIME_PERIOD', 
                      'DAY_OF_WEEK', 'DELAY_CATEGORY']].copy()

def get_season(m):
    if m in [12, 1, 2]: return 'Winter'
    elif m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    else: return 'Fall'

bn_df['SEASON'] = bn_df['MONTH'].apply(get_season)
bn_df.drop(columns=['MONTH'], inplace=True)

def get_day_type(d):
    return 'Weekend' if d in [6, 7] else 'Weekday'

bn_df['DAY_TYPE'] = bn_df['DAY_OF_WEEK'].apply(get_day_type)
bn_df.drop(columns=['DAY_OF_WEEK'], inplace=True)

print(f"Final BN dataset shape: {bn_df.shape}")
print(bn_df.head())

# ── Split ────────────────────────────────────────────────────────
train_df, test_df = train_test_split(
    bn_df, test_size=0.2, random_state=42,
    stratify=bn_df['DELAY_CATEGORY']
)
print(f"\nTrain: {train_df.shape} | Test: {test_df.shape}")

# ── Define DAG ───────────────────────────────────────────────────
model = DiscreteBayesianNetwork([
    ('AIRLINE',      'DELAY_CATEGORY'),
    ('SEASON',       'DELAY_CATEGORY'),
    ('TIME_PERIOD',  'DELAY_CATEGORY'),
    ('DAY_TYPE',     'DELAY_CATEGORY'),
])

# ── Train ────────────────────────────────────────────────────────
estimator = DiscreteBayesianEstimator(prior_type='BDeu', equivalent_sample_size=10)
model.fit(train_df, estimator=estimator)
print("\nModel trained.")
print(f"Nodes: {list(model.nodes())}")

# ── Evaluate with progress bar ────────────────────────────────────
inference = VariableElimination(model)
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Running inference"):
    result = inference.query(
        variables=['DELAY_CATEGORY'],
        evidence={
            'AIRLINE': row['AIRLINE'],
            'SEASON': row['SEASON'],
            'TIME_PERIOD': row['TIME_PERIOD'],
            'DAY_TYPE': row['DAY_TYPE'],
        },
        show_progress=False
    )
    states = result.state_names['DELAY_CATEGORY']
    predictions.append(states[result.values.argmax()])

y_true = test_df['DELAY_CATEGORY'].tolist()

print(f"\nAccuracy: {accuracy_score(y_true, predictions):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_true, predictions))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_true, predictions, labels=['OnTime','Short','Long']))

# ── Save ─────────────────────────────────────────────────────────
with open('flight_delay_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("\nSaved: flight_delay_model.pkl")

Final BN dataset shape: (300000, 5)
        AIRLINE TIME_PERIOD DELAY_CATEGORY  SEASON DAY_TYPE
2978953      OO   Afternoon           Long  Summer  Weekday
5075147      AS   Afternoon           Long    Fall  Weekday
3570064      MQ   Afternoon           Long  Summer  Weekday
55217        EV   Afternoon           Long  Winter  Weekend
5703841      WN     Morning           Long  Winter  Weekday

Train: (240000, 5) | Test: (60000, 5)

Model trained.
Nodes: ['AIRLINE', 'DELAY_CATEGORY', 'SEASON', 'TIME_PERIOD', 'DAY_TYPE']


Running inference: 100%|██████████| 60000/60000 [00:35<00:00, 1674.85it/s]



Accuracy: 0.4341

Classification Report:
              precision    recall  f1-score   support

        Long       0.41      0.51      0.46     20000
      OnTime       0.47      0.52      0.49     20000
       Short       0.41      0.27      0.33     20000

    accuracy                           0.43     60000
   macro avg       0.43      0.43      0.43     60000
weighted avg       0.43      0.43      0.43     60000


Confusion Matrix:
[[10437  3515  6048]
 [ 6101  5385  8514]
 [ 5641  4137 10222]]

Saved: flight_delay_model.pkl


In [9]:
# Show actual probability shifts across real scenarios — 
# this is more honest and more useful than the accuracy number

scenarios = [
    ('Best case: Alaska, Morning, Weekday, Fall', 
     {'AIRLINE':'AS','TIME_PERIOD':'Morning','DAY_TYPE':'Weekday','SEASON':'Fall'}),
    ('Worst case: JetBlue, Evening, Weekday, Summer',
     {'AIRLINE':'B6','TIME_PERIOD':'Evening','DAY_TYPE':'Weekday','SEASON':'Summer'}),
    ('Mixed: Delta, Afternoon, Weekend, Winter',
     {'AIRLINE':'DL','TIME_PERIOD':'Afternoon','DAY_TYPE':'Weekend','SEASON':'Winter'}),
]

for label, ev in scenarios:
    r = inference.query(variables=['DELAY_CATEGORY'], evidence=ev, show_progress=False)
    states = r.state_names['DELAY_CATEGORY']
    print(f"\n{label}")
    for s, p in zip(states, r.values):
        print(f"  {s}: {p*100:.1f}%")


Best case: Alaska, Morning, Weekday, Fall
  Long: 19.0%
  OnTime: 60.4%
  Short: 20.6%

Worst case: JetBlue, Evening, Weekday, Summer
  Long: 48.5%
  OnTime: 16.9%
  Short: 34.5%

Mixed: Delta, Afternoon, Weekend, Winter
  Long: 29.1%
  OnTime: 38.5%
  Short: 32.4%


In [10]:
import pickle
import warnings
warnings.filterwarnings('ignore')

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm
import pandas as pd

# ══════════════════════════════════════════════════════════════════
# STEP 1 — REBUILD WITH ORIGIN AIRPORT + BINARY TARGET
# ══════════════════════════════════════════════════════════════════

# Start from df_balanced (already balanced 3-class) — but we need
# origin airport, which wasn't in bn_df. Pull it back in.
cols = ['AIRLINE', 'MONTH', 'TIME_PERIOD', 'DAY_OF_WEEK', 
        'ORIGIN_AIRPORT', 'DELAY_CATEGORY']
bn_df = df_balanced[cols].copy()

def get_season(m):
    if m in [12, 1, 2]: return 'Winter'
    elif m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    else: return 'Fall'
bn_df['SEASON'] = bn_df['MONTH'].apply(get_season)
bn_df.drop(columns=['MONTH'], inplace=True)

def get_day_type(d):
    return 'Weekend' if d in [6, 7] else 'Weekday'
bn_df['DAY_TYPE'] = bn_df['DAY_OF_WEEK'].apply(get_day_type)
bn_df.drop(columns=['DAY_OF_WEEK'], inplace=True)

# Keep only top 15 busiest origin airports — rest become 'Other'
# (too many airports = sparse CPTs, this keeps it clean)
top_airports = bn_df['ORIGIN_AIRPORT'].value_counts().head(15).index.tolist()
bn_df['ORIGIN'] = bn_df['ORIGIN_AIRPORT'].apply(
    lambda x: x if x in top_airports else 'Other'
)
bn_df.drop(columns=['ORIGIN_AIRPORT'], inplace=True)

# Binary target — OnTime vs Delayed
bn_df['DELAY_BINARY'] = bn_df['DELAY_CATEGORY'].apply(
    lambda x: 'OnTime' if x == 'OnTime' else 'Delayed'
)
bn_df.drop(columns=['DELAY_CATEGORY'], inplace=True)

print(f"Shape: {bn_df.shape}")
print(f"\nBinary target distribution:\n{bn_df['DELAY_BINARY'].value_counts()}")
print(f"\nTop airports used:\n{bn_df['ORIGIN'].value_counts()}")

# ══════════════════════════════════════════════════════════════════
# STEP 2 — SPLIT
# ══════════════════════════════════════════════════════════════════

train_df, test_df = train_test_split(
    bn_df, test_size=0.2, random_state=42,
    stratify=bn_df['DELAY_BINARY']
)
print(f"\nTrain: {train_df.shape} | Test: {test_df.shape}")

# ══════════════════════════════════════════════════════════════════
# STEP 3 — DEFINE + TRAIN
# ══════════════════════════════════════════════════════════════════

model = DiscreteBayesianNetwork([
    ('AIRLINE',      'DELAY_BINARY'),
    ('SEASON',       'DELAY_BINARY'),
    ('TIME_PERIOD',  'DELAY_BINARY'),
    ('DAY_TYPE',     'DELAY_BINARY'),
    ('ORIGIN',       'DELAY_BINARY'),
])

estimator = DiscreteBayesianEstimator(prior_type='BDeu', equivalent_sample_size=10)
model.fit(train_df, estimator=estimator)
print("\nModel trained.")
print(f"Nodes: {list(model.nodes())}")

# ══════════════════════════════════════════════════════════════════
# STEP 4 — EVALUATE
# ══════════════════════════════════════════════════════════════════

inference = VariableElimination(model)
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Running inference"):
    result = inference.query(
        variables=['DELAY_BINARY'],
        evidence={
            'AIRLINE': row['AIRLINE'],
            'SEASON': row['SEASON'],
            'TIME_PERIOD': row['TIME_PERIOD'],
            'DAY_TYPE': row['DAY_TYPE'],
            'ORIGIN': row['ORIGIN'],
        },
        show_progress=False
    )
    states = result.state_names['DELAY_BINARY']
    predictions.append(states[result.values.argmax()])

y_true = test_df['DELAY_BINARY'].tolist()

print(f"\nAccuracy: {accuracy_score(y_true, predictions):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_true, predictions))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_true, predictions, labels=['OnTime','Delayed']))

# ══════════════════════════════════════════════════════════════════
# STEP 5 — SCENARIO PROOF (same style as before)
# ══════════════════════════════════════════════════════════════════

print("\n" + "="*50)
print("SCENARIO TEST")
print("="*50)

scenarios = [
    ('Best case: Alaska, Morning, Weekday, Fall, Small airport',
     {'AIRLINE':'AS','TIME_PERIOD':'Morning','DAY_TYPE':'Weekday',
      'SEASON':'Fall','ORIGIN':'Other'}),
    ('Worst case: JetBlue, Evening, Weekday, Summer, Big hub',
     {'AIRLINE':'B6','TIME_PERIOD':'Evening','DAY_TYPE':'Weekday',
      'SEASON':'Summer','ORIGIN':bn_df['ORIGIN'].value_counts().index[0]}),
]

for label, ev in scenarios:
    r = inference.query(variables=['DELAY_BINARY'], evidence=ev, show_progress=False)
    states = r.state_names['DELAY_BINARY']
    print(f"\n{label}")
    for s, p in zip(states, r.values):
        print(f"  {s}: {p*100:.1f}%")

# ══════════════════════════════════════════════════════════════════
# STEP 6 — SAVE
# ══════════════════════════════════════════════════════════════════

with open('flight_delay_model_v2.pkl', 'wb') as f:
    pickle.dump(model, f)
print("\nSaved: flight_delay_model_v2.pkl")

Shape: (300000, 6)

Binary target distribution:
DELAY_BINARY
Delayed    200000
OnTime     100000
Name: count, dtype: int64

Top airports used:
ORIGIN
Other    162125
ORD       17788
ATL       17681
DFW       13703
DEN       11507
LAX       10854
SFO        8422
IAH        8265
LAS        7573
PHX        7359
MCO        6345
EWR        6087
LGA        5799
BOS        5624
DTW        5533
JFK        5335
Name: count, dtype: int64

Train: (240000, 6) | Test: (60000, 6)

Model trained.
Nodes: ['AIRLINE', 'DELAY_BINARY', 'SEASON', 'TIME_PERIOD', 'DAY_TYPE', 'ORIGIN']


Running inference: 100%|██████████| 60000/60000 [00:37<00:00, 1615.48it/s]



Accuracy: 0.6832

Classification Report:
              precision    recall  f1-score   support

     Delayed       0.70      0.90      0.79     40000
      OnTime       0.56      0.24      0.34     20000

    accuracy                           0.68     60000
   macro avg       0.63      0.57      0.56     60000
weighted avg       0.66      0.68      0.64     60000


Confusion Matrix:
[[ 4849 15151]
 [ 3856 36144]]

SCENARIO TEST

Best case: Alaska, Morning, Weekday, Fall, Small airport
  Delayed: 38.9%
  OnTime: 61.1%

Worst case: JetBlue, Evening, Weekday, Summer, Big hub
  Delayed: 81.9%
  OnTime: 18.1%

Saved: flight_delay_model_v2.pkl


In [11]:
import pandas as pd

# Work with a fresh sample from the original flights data (not the balanced BN one — 
# we need real chronological sequences per aircraft)
flights_seq = flights[flights['CANCELLED'] == 0][
    ['TAIL_NUMBER','MONTH','DAY','SCHEDULED_DEPARTURE','DEPARTURE_DELAY']
].dropna()

# Sort by aircraft, then by day, then by scheduled time — gives true flight order
flights_seq = flights_seq.sort_values(['TAIL_NUMBER','MONTH','DAY','SCHEDULED_DEPARTURE'])

# Categorize delay same as before
def categorize_delay(d):
    if d <= 15: return 'OnTime'
    else: return 'Delayed'

flights_seq['DELAY_CAT'] = flights_seq['DEPARTURE_DELAY'].apply(categorize_delay)

# Shift within each tail number group to get "next flight's delay"
flights_seq['NEXT_DELAY_CAT'] = flights_seq.groupby('TAIL_NUMBER')['DELAY_CAT'].shift(-1)

# Drop rows where there's no next flight (last flight of the day for that aircraft)
cascade_df = flights_seq.dropna(subset=['NEXT_DELAY_CAT'])

print(f"Cascade pairs: {cascade_df.shape}")
print(f"\nDoes current delay predict next delay?")
print(pd.crosstab(cascade_df['DELAY_CAT'], cascade_df['NEXT_DELAY_CAT'], normalize='index'))

Cascade pairs: (5724299, 7)

Does current delay predict next delay?
NEXT_DELAY_CAT   Delayed    OnTime
DELAY_CAT                         
Delayed         0.482899  0.517101
OnTime          0.111564  0.888436


In [12]:
import pickle
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator
from pgmpy.inference import VariableElimination
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm

# ── Prepare cascade dataset ─────────────────────────────────────
cascade_bn_df = cascade_df[['DELAY_CAT', 'NEXT_DELAY_CAT']].copy()
cascade_bn_df.columns = ['CURRENT_DELAY', 'NEXT_DELAY']

# Balance it — same discipline as before, real undersampling
min_size = cascade_bn_df.groupby(['CURRENT_DELAY']).size().min()
sample_size = min(min_size, 150000)

cascade_balanced = (
    cascade_bn_df.groupby('CURRENT_DELAY', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), sample_size), random_state=42))
)

print(f"Cascade training shape: {cascade_balanced.shape}")
print(cascade_balanced['CURRENT_DELAY'].value_counts())

# ── Split ─────────────────────────────────────────────────────
train_c, test_c = train_test_split(
    cascade_balanced, test_size=0.2, random_state=42,
    stratify=cascade_balanced['CURRENT_DELAY']
)

# ── Define + train cascade BN ───────────────────────────────────
cascade_model = DiscreteBayesianNetwork([
    ('CURRENT_DELAY', 'NEXT_DELAY')
])

est = DiscreteBayesianEstimator(prior_type='BDeu', equivalent_sample_size=10)
cascade_model.fit(train_c, estimator=est)
print("\nCascade model trained.")

# ── Evaluate ─────────────────────────────────────────────────────
inference_c = VariableElimination(cascade_model)
preds = []
for _, row in tqdm(test_c.iterrows(), total=len(test_c), desc="Cascade inference"):
    r = inference_c.query(variables=['NEXT_DELAY'], 
                          evidence={'CURRENT_DELAY': row['CURRENT_DELAY']}, 
                          show_progress=False)
    states = r.state_names['NEXT_DELAY']
    preds.append(states[r.values.argmax()])

y_true_c = test_c['NEXT_DELAY'].tolist()
print(f"\nCascade Accuracy: {accuracy_score(y_true_c, preds):.4f}")
print(classification_report(y_true_c, preds))

# ── Live cascade query — the actual demo moment ──────────────────
print("\n" + "="*50)
print("CASCADE DEMO")
print("="*50)
for current in ['OnTime', 'Delayed']:
    r = inference_c.query(variables=['NEXT_DELAY'], 
                          evidence={'CURRENT_DELAY': current}, 
                          show_progress=False)
    print(f"\nIf current flight is {current}:")
    for s, p in zip(r.state_names['NEXT_DELAY'], r.values):
        print(f"  Next flight {s}: {p*100:.1f}%")

# ── Save ─────────────────────────────────────────────────────────
with open('cascade_model.pkl', 'wb') as f:
    pickle.dump(cascade_model, f)
print("\nSaved: cascade_model.pkl")

Cascade training shape: (300000, 2)
CURRENT_DELAY
Delayed    150000
OnTime     150000
Name: count, dtype: int64

Cascade model trained.


Cascade inference: 100%|██████████| 60000/60000 [00:27<00:00, 2180.32it/s]



Cascade Accuracy: 0.7010
              precision    recall  f1-score   support

     Delayed       0.00      0.00      0.00     17937
      OnTime       0.70      1.00      0.82     42063

    accuracy                           0.70     60000
   macro avg       0.35      0.50      0.41     60000
weighted avg       0.49      0.70      0.58     60000


CASCADE DEMO

If current flight is OnTime:
  Next flight Delayed: 11.2%
  Next flight OnTime: 88.8%

If current flight is Delayed:
  Next flight Delayed: 48.2%
  Next flight OnTime: 51.8%

Saved: cascade_model.pkl


In [14]:
# Reverse query — this is the "no classifier can do this" demo moment
print("BACKWARD DIAGNOSTIC REASONING")
print("="*50)

r = inference_c.query(variables=['CURRENT_DELAY'], 
                      evidence={'NEXT_DELAY': 'Delayed'}, 
                      show_progress=False)
print("\nGiven the next flight WAS delayed, what caused it?")
for s, p in zip(r.state_names['CURRENT_DELAY'], r.values):
    print(f"  Previous flight was {s}: {p*100:.1f}%")
    

BACKWARD DIAGNOSTIC REASONING

Given the next flight WAS delayed, what caused it?
  Previous flight was Delayed: 81.2%
  Previous flight was OnTime: 18.8%


In [15]:
import numpy as np

def entropy(probs):
    """Shannon entropy — measures uncertainty. Lower = more certain."""
    probs = np.array(probs)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

# Base case — no evidence at all, maximum uncertainty
inference = VariableElimination(model)  # your base model from Phase 3

r0 = inference.query(variables=['DELAY_BINARY'], show_progress=False)
base_entropy = entropy(r0.values)
print(f"Baseline uncertainty (no evidence): {base_entropy:.4f} bits")
print(f"  Delayed: {r0.values[0]*100:.1f}%  OnTime: {r0.values[1]*100:.1f}%")

# Now test: if we learned ONLY the airline, how much does uncertainty drop?
# vs if we learned ONLY the origin airport?
# vs ONLY the time period?

test_variables = ['AIRLINE', 'ORIGIN', 'TIME_PERIOD', 'SEASON', 'DAY_TYPE']

print("\n" + "="*50)
print("VALUE OF INFORMATION — which factor reduces uncertainty most?")
print("="*50)

for var in test_variables:
    # average entropy across all possible values of this variable,
    # weighted by how often each value occurs
    possible_values = train_df[var].unique()
    weighted_entropy = 0
    total_weight = len(train_df)
    
    for val in possible_values:
        weight = (train_df[var] == val).sum()
        if weight == 0:
            continue
        r = inference.query(variables=['DELAY_BINARY'], 
                            evidence={var: val}, show_progress=False)
        weighted_entropy += (weight/total_weight) * entropy(r.values)
    
    info_gain = base_entropy - weighted_entropy
    print(f"\n{var}:")
    print(f"  Average uncertainty after knowing this: {weighted_entropy:.4f} bits")
    print(f"  Information gain: {info_gain:.4f} bits")

Baseline uncertainty (no evidence): 0.9415 bits
  Delayed: 64.1%  OnTime: 35.9%

VALUE OF INFORMATION — which factor reduces uncertainty most?

AIRLINE:
  Average uncertainty after knowing this: 0.9354 bits
  Information gain: 0.0061 bits

ORIGIN:
  Average uncertainty after knowing this: 0.9404 bits
  Information gain: 0.0011 bits

TIME_PERIOD:
  Average uncertainty after knowing this: 0.9207 bits
  Information gain: 0.0209 bits

SEASON:
  Average uncertainty after knowing this: 0.9360 bits
  Information gain: 0.0055 bits

DAY_TYPE:
  Average uncertainty after knowing this: 0.9413 bits
  Information gain: 0.0003 bits
